# Tarea 5: Seguimiento
## Notebook 07 — Framework de KPIs

**Objetivo:**

Definir un framework de KPIs en dos niveles:

1. **KPIs de Campaña** — Para medir el rendimiento de la campaña de email marketing (9.836 clientes, recomendación de `credit_card`).
2. **KPIs Estratégicos** — Para hacer seguimiento de la nueva estrategia de penetración de cartera (Matriz de Ansoff: mercados actuales × productos actuales).

**Nota metodológica:**  
Los KPIs de campaña combinan métricas calculadas sobre los datos disponibles con benchmarks estándar del sector. Los resultados post-envío se ilustran con valores ficticios representativos para mostrar la metodología de medición.

**Input:**  
- `recomendacion_10000_personalizado.csv`  
- `master_df_flags.parquet`

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All imports OK")

---
# 1. Carga de datos

In [23]:
# Cargar datos de la campaña (output de Tarea 4 - Personalización)
df_camp = pd.read_csv('..\\Tarea 04 Personalización\\recomendacion_10000_personalizado.csv')
print(f"Clientes en campaña: {len(df_camp):,}")
print(df_camp.head(3))
print(f"\nColumnas: {df_camp.columns.tolist()}")

Clientes en campaña: 9,836
   id_user recommended_product  creatividad_id   nombre_creatividad  \
0  1114367         credit_card               5  Premium y exclusivo   
1  1070525         credit_card               5  Premium y exclusivo   
2  1437442         credit_card               5  Premium y exclusivo   

                                             mensaje  
0  Accede a un mundo de ventajas exclusivas. Lími...  
1  Accede a un mundo de ventajas exclusivas. Lími...  
2  Accede a un mundo de ventajas exclusivas. Lími...  

Columnas: ['id_user', 'recommended_product', 'creatividad_id', 'nombre_creatividad', 'mensaje']


In [24]:
# Cargar datos maestros para KPIs estratégicos
df_master = pd.read_parquet('../../data/processed/master_df_flags.parquet')

# Filtrar anomalías (igual que en tareas anteriores)
anomaly_cols = ['age_anomaly', 'deceased_anomaly', 'entry_date_anomaly', 'salary_anomaly']
existing_anomaly = [c for c in anomaly_cols if c in df_master.columns]
if existing_anomaly:
    df_master = df_master[(df_master[existing_anomaly] == 0).all(axis=1)].copy()

# Última partición = estado actual de clientes
ultima_particion = df_master['pk_partition'].max()
df_actual = df_master[df_master['pk_partition'] == ultima_particion].copy()

print(f"Registros maestros (sin anomalías): {len(df_master):,}")
print(f"Última partición: {ultima_particion}")
print(f"Clientes en estado actual: {len(df_actual):,}")

Registros maestros (sin anomalías): 5,940,692
Última partición: 2019-05-28 00:00:00
Clientes en estado actual: 441,752


---
# 2. KPIs de Campaña

## 2.1 Definición del framework

La campaña consiste en enviar a **9.836 clientes** una recomendación personalizada de `credit_card` con 5 creatividades distintas según su perfil sociodemográfico.

El funnel de conversión de email marketing tiene las siguientes etapas:

ENVIADOS (9.836)
→ ENTREGADOS  (Delivery Rate)
→ ABIERTOS  (Open Rate)
→ CLICKS  (Click-Through Rate)
→ CONVERSIONES  (Conversion Rate)
→ REVENUE  (ROI)

**Márgenes por producto (fuente: briefing del proyecto):**
- Tarjetas de crédito/débito: **60 €** por contrato
- Productos ahorro/inversión (fondos, planes, depósitos): **40 €**
- Cuentas: **10 €**

**Costo estimado por email:** ~0,01 € (ESP estándar)

> ⚠️ Los resultados post-envío son valores ficticios ilustrativos. En producción vendrán del ESP (Email Service Provider).

In [25]:
# ============================================================
# PARÁMETROS DE LA CAMPAÑA
# ============================================================

N_ENVIADOS = len(df_camp)
MARGEN_CREDIT_CARD = 60  # €
COSTO_EMAIL = 0.01       # € por email

# Benchmarks del sector para email marketing financiero
# (fuente: Campaign Monitor / Mailchimp Industry Benchmarks 2023)
BENCHMARK_DELIVERY_RATE = 0.98
BENCHMARK_OPEN_RATE     = 0.22
BENCHMARK_CTR           = 0.028
BENCHMARK_CONV_RATE     = 0.015

print("Parámetros definidos:")
print(f"  Emails a enviar:    {N_ENVIADOS:,}")
print(f"  Margen credit card: {MARGEN_CREDIT_CARD} €")
print(f"  Costo por email:    {COSTO_EMAIL} €")
print(f"  Costo total:        {N_ENVIADOS * COSTO_EMAIL:.0f} €")
print(f"\nBenchmarks del sector:")
print(f"  Delivery Rate: {BENCHMARK_DELIVERY_RATE:.0%}")
print(f"  Open Rate:     {BENCHMARK_OPEN_RATE:.0%}")
print(f"  CTR:           {BENCHMARK_CTR:.1%}")
print(f"  Conv Rate:     {BENCHMARK_CONV_RATE:.1%}")

Parámetros definidos:
  Emails a enviar:    9,836
  Margen credit card: 60 €
  Costo por email:    0.01 €
  Costo total:        98 €

Benchmarks del sector:
  Delivery Rate: 98%
  Open Rate:     22%
  CTR:           2.8%
  Conv Rate:     1.5%


In [26]:
# Distribución de clientes por creatividad
dist_creatividades = df_camp.groupby(['creatividad_id', 'nombre_creatividad']).size().reset_index(name='n_clientes')
dist_creatividades['pct'] = dist_creatividades['n_clientes'] / N_ENVIADOS

print("Distribución de clientes por creatividad:")
print(dist_creatividades.to_string(index=False))

Distribución de clientes por creatividad:
 creatividad_id     nombre_creatividad  n_clientes      pct
              1  Experiencia y solidez        1363 0.138573
              2      Éxito sin límites        3174 0.322692
              3 Inteligente y práctico        1119 0.113766
              4     Activo y conectado        2385 0.242477
              5    Premium y exclusivo        1795 0.182493


In [27]:
# ============================================================
# SIMULACIÓN DEL FUNNEL POR CREATIVIDAD (valores ficticios ilustrativos)
# ============================================================

# Asignamos tasas ligeramente distintas por creatividad
# para reflejar que diferentes mensajes tienen diferente rendimiento
creatividades_params = {
    1: {"nombre": "Experiencia y solidez",   "open_rate": 0.198, "ctr": 0.024, "conv_rate": 0.013},
    2: {"nombre": "Éxito sin límites",       "open_rate": 0.241, "ctr": 0.031, "conv_rate": 0.017},
    3: {"nombre": "Inteligente y práctico",  "open_rate": 0.235, "ctr": 0.029, "conv_rate": 0.016},
    4: {"nombre": "Activo y conectado",      "open_rate": 0.254, "ctr": 0.034, "conv_rate": 0.019},
    5: {"nombre": "Premium y exclusivo",     "open_rate": 0.268, "ctr": 0.038, "conv_rate": 0.022},
}

rows = []
for _, row_camp in dist_creatividades.iterrows():
    cid    = row_camp['creatividad_id']
    n      = row_camp['n_clientes']
    params = creatividades_params[cid]

    entregados   = int(n * BENCHMARK_DELIVERY_RATE)
    abiertos     = int(entregados * params['open_rate'])
    clicks       = int(entregados * params['ctr'])
    conversiones = int(n * params['conv_rate'])
    revenue      = conversiones * MARGEN_CREDIT_CARD
    costo        = n * COSTO_EMAIL
    roi_pct      = ((revenue - costo) / costo) * 100

    rows.append({
        'Creatividad':    params['nombre'],
        'Enviados':       n,
        'Entregados':     entregados,
        'Delivery Rate':  entregados / n,
        'Abiertos':       abiertos,
        'Open Rate':      abiertos / entregados,
        'Clicks':         clicks,
        'CTR':            clicks / entregados,
        'Conversiones':   conversiones,
        'Conv Rate':      conversiones / n,
        'Revenue (€)':    revenue,
        'Costo (€)':      costo,
        'ROI (%)':        roi_pct,
    })

df_kpi_camp = pd.DataFrame(rows)

# Fila de totales
total = {
    'Creatividad':   'TOTAL',
    'Enviados':      df_kpi_camp['Enviados'].sum(),
    'Entregados':    df_kpi_camp['Entregados'].sum(),
    'Delivery Rate': df_kpi_camp['Entregados'].sum() / df_kpi_camp['Enviados'].sum(),
    'Abiertos':      df_kpi_camp['Abiertos'].sum(),
    'Open Rate':     df_kpi_camp['Abiertos'].sum() / df_kpi_camp['Entregados'].sum(),
    'Clicks':        df_kpi_camp['Clicks'].sum(),
    'CTR':           df_kpi_camp['Clicks'].sum() / df_kpi_camp['Entregados'].sum(),
    'Conversiones':  df_kpi_camp['Conversiones'].sum(),
    'Conv Rate':     df_kpi_camp['Conversiones'].sum() / df_kpi_camp['Enviados'].sum(),
    'Revenue (€)':   df_kpi_camp['Revenue (€)'].sum(),
    'Costo (€)':     df_kpi_camp['Costo (€)'].sum(),
    'ROI (%)':       ((df_kpi_camp['Revenue (€)'].sum() - df_kpi_camp['Costo (€)'].sum()) / df_kpi_camp['Costo (€)'].sum()) * 100,
}
df_kpi_camp = pd.concat([df_kpi_camp, pd.DataFrame([total])], ignore_index=True)

# Mostrar tabla resumen
cols_show = ['Creatividad', 'Enviados', 'Open Rate', 'CTR', 'Conv Rate', 'Conversiones', 'Revenue (€)', 'ROI (%)']
print("📊 KPIs de Campaña por Creatividad:")
print(df_kpi_camp[cols_show].to_string(index=False))

📊 KPIs de Campaña por Creatividad:
           Creatividad  Enviados  Open Rate      CTR  Conv Rate  Conversiones  Revenue (€)      ROI (%)
 Experiencia y solidez      1363   0.197753 0.023970   0.012472            17         1020  7383.492296
     Éxito sin límites      3174   0.240836 0.030868   0.016698            53         3180  9918.903592
Inteligente y práctico      1119   0.234489 0.028285   0.015192            17         1020  9015.281501
    Activo y conectado      2385   0.253744 0.033804   0.018868            45         2700 11220.754717
   Premium y exclusivo      1795   0.267766 0.037521   0.021727            39         2340 12936.211699
                 TOTAL      9836   0.242192 0.031545   0.017385           171        10260 10331.069540


## 2.2 Visualización — Funnel y KPIs por creatividad

In [28]:
# ============================================================
# FUNNEL GLOBAL DE CAMPAÑA
# ============================================================

total_row = df_kpi_camp[df_kpi_camp['Creatividad'] == 'TOTAL'].iloc[0]

fig_funnel = go.Figure(go.Funnel(
    y = ['Enviados', 'Entregados', 'Abiertos', 'Clicks', 'Conversiones'],
    x = [
        int(total_row['Enviados']),
        int(total_row['Entregados']),
        int(total_row['Abiertos']),
        int(total_row['Clicks']),
        int(total_row['Conversiones']),
    ],
    textinfo = 'value+percent initial',
    marker = dict(color=['#2196F3', '#42A5F5', '#66BB6A', '#FFA726', '#EF5350']),
    connector = {'line': {'color': '#BDBDBD', 'width': 1}}
))

fig_funnel.update_layout(
    title=dict(text='Funnel de Conversión — Campaña easyMoney (Credit Card)',
               x=0.5, font=dict(size=16)),
    template='plotly_white',
    width=700, height=450
)
fig_funnel.show()

print(f"\n💰 Revenue total estimado: {int(total_row['Revenue (€)']):,} €")
print(f"📧 Costo total campaña:    {total_row['Costo (€)']:.0f} €")
print(f"📈 ROI:                    {total_row['ROI (%)']:.0f}%")


💰 Revenue total estimado: 10,260 €
📧 Costo total campaña:    98 €
📈 ROI:                    10331%


El funnel confirma la eficiencia de la campaña: de los **9.836 emails enviados** se estiman **171 contratos** firmados, generando **10.260 € de revenue** con un coste de apenas **98 €**. El ROI del 10.331% es excepcional incluso bajo supuestos conservadores, lo que valida la viabilidad económica de la estrategia de recomendación personalizada.

In [29]:
# ============================================================
# COMPARATIVA DE KPIs POR CREATIVIDAD
# ============================================================

df_plot = df_kpi_camp[df_kpi_camp['Creatividad'] != 'TOTAL'].copy()
colors = ['#2196F3', '#66BB6A', '#FFA726', '#EF5350', '#AB47BC']

fig_comp = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Open Rate por Creatividad',
        'CTR por Creatividad',
        'Tasa de Conversión',
        'Revenue (€) por Creatividad'
    ],
    vertical_spacing=0.22,
    horizontal_spacing=0.12
)

# Open Rate
fig_comp.add_trace(go.Bar(
    x=df_plot['Creatividad'], y=df_plot['Open Rate'],
    marker_color=colors,
    text=[f"{v:.1%}" for v in df_plot['Open Rate']],
    textposition='outside'
), row=1, col=1)

# CTR
fig_comp.add_trace(go.Bar(
    x=df_plot['Creatividad'], y=df_plot['CTR'],
    marker_color=colors,
    text=[f"{v:.1%}" for v in df_plot['CTR']],
    textposition='outside'
), row=1, col=2)

# Conv Rate
fig_comp.add_trace(go.Bar(
    x=df_plot['Creatividad'], y=df_plot['Conv Rate'],
    marker_color=colors,
    text=[f"{v:.1%}" for v in df_plot['Conv Rate']],
    textposition='outside'
), row=2, col=1)

# Revenue
fig_comp.add_trace(go.Bar(
    x=df_plot['Creatividad'], y=df_plot['Revenue (€)'],
    marker_color=colors,
    text=[f"{int(v):,} €" for v in df_plot['Revenue (€)']],
    textposition='outside'
), row=2, col=2)

fig_comp.update_layout(
    title=dict(text='KPIs de Campaña por Creatividad', x=0.5, font=dict(size=16)),
    template='plotly_white',
    showlegend=False,
    width=1000, height=620
)
fig_comp.update_xaxes(tickangle=-25)
fig_comp.show()

La creatividad **"Premium y exclusivo"** lidera en todos los indicadores (Open Rate 26.8%, Conv Rate 2.2%, Revenue 2.340 €), lo que confirma que el segmento de mayor renta responde mejor a mensajes de exclusividad. Por el contrario, **"Experiencia y solidez"** obtiene los resultados más bajos en conversión (1.3%), lo que sugiere que el mensaje orientado a seniority puede requerir ajuste de tono en futuras iteraciones de la campaña.

## 2.3 Benchmark vs Resultado

Comparamos los resultados globales de la campaña con los benchmarks del sector financiero.

In [30]:
# ============================================================
# BENCHMARK vs RESULTADO GLOBAL
# ============================================================

total_row = df_kpi_camp[df_kpi_camp['Creatividad'] == 'TOTAL'].iloc[0]

kpis_bench = ['Open Rate', 'CTR', 'Conv Rate']
benchmarks  = [BENCHMARK_OPEN_RATE, BENCHMARK_CTR, BENCHMARK_CONV_RATE]
campana     = [total_row['Open Rate'], total_row['CTR'], total_row['Conv Rate']]

fig_bench = go.Figure()

fig_bench.add_trace(go.Bar(
    name='Benchmark sector',
    x=kpis_bench,
    y=benchmarks,
    marker_color='#BDBDBD',
    text=[f"{v:.1%}" for v in benchmarks],
    textposition='outside'
))

fig_bench.add_trace(go.Bar(
    name='Campaña easyMoney',
    x=kpis_bench,
    y=campana,
    marker_color='#66BB6A',
    text=[f"{v:.1%}" for v in campana],
    textposition='outside'
))

fig_bench.update_layout(
    title=dict(text='Campaña easyMoney vs Benchmark del Sector Financiero',
               x=0.5, font=dict(size=15)),
    barmode='group',
    template='plotly_white',
    legend=dict(x=0.75, y=0.95),
    width=750, height=420
)
fig_bench.show()

# Resumen textual
print("\nResumen comparativo:")
for kpi, bench, camp in zip(kpis_bench, benchmarks, campana):
    icono = "✅" if camp > bench else "⚠️"
    delta = camp - bench
    print(f"  {icono} {kpi}: Campaña {camp:.1%} vs Benchmark {bench:.1%} (Δ {delta:+.1%})")


Resumen comparativo:
  ✅ Open Rate: Campaña 24.2% vs Benchmark 22.0% (Δ +2.2%)
  ✅ CTR: Campaña 3.2% vs Benchmark 2.8% (Δ +0.4%)
  ✅ Conv Rate: Campaña 1.7% vs Benchmark 1.5% (Δ +0.2%)


---
# 3. KPIs Estratégicos

## 3.1 Contexto

La nueva estrategia de easyMoney se centra en **penetración de mercado** (cuadrante inferior-izquierdo de la Matriz de Ansoff):

> *"Obtener una mejor rentabilidad de la base actual de clientes"*

Para medir el progreso de esta estrategia definimos KPIs en cinco dimensiones:

| Dimensión | ¿Qué medimos? |
|-----------|---------------|
| **Profundidad de cartera** | ¿Cuántos productos tiene cada cliente? |
| **Activación** | ¿Qué porcentaje de clientes está activo en la app? |
| **Cross-sell** | ¿A qué tasa vendemos productos adicionales? |
| **Rentabilidad** | ¿Cuánto ingresamos por cliente? |
| **Retención** | ¿Perdemos clientes mes a mes? |

In [31]:
# ============================================================
# COLUMNAS DE PRODUCTOS DISPONIBLES
# ============================================================

product_cols = [
    'em_acount', 'em_account_p', 'em_account_pp', 'emc_account',
    'payroll_account', 'credit_card', 'debit_card',
    'funds', 'long_term_deposit', 'short_term_deposit',
    'pension_plan', 'securities', 'loans', 'mortgage', 'payroll'
]

margenes = {
    'credit_card': 60, 'debit_card': 60,
    'loans': 60, 'mortgage': 60,
    'funds': 40, 'long_term_deposit': 40,
    'short_term_deposit': 40, 'pension_plan': 40, 'securities': 40,
    'em_acount': 10, 'em_account_p': 10, 'em_account_pp': 10,
    'emc_account': 10, 'payroll_account': 10, 'payroll': 10
}

existing_prods = [c for c in product_cols if c in df_actual.columns]
print(f"Productos disponibles en los datos: {len(existing_prods)}")
print(existing_prods)

Productos disponibles en los datos: 14
['em_acount', 'em_account_p', 'emc_account', 'payroll_account', 'credit_card', 'debit_card', 'funds', 'long_term_deposit', 'short_term_deposit', 'pension_plan', 'securities', 'loans', 'mortgage', 'payroll']


In [32]:
# ============================================================
# CÁLCULO DE KPIs ESTRATÉGICOS — ESTADO ACTUAL
# ============================================================

df_actual = df_actual.copy()

# Número de productos por cliente
df_actual['n_productos'] = df_actual[existing_prods].sum(axis=1)

# Revenue estimado por cliente
df_actual['revenue_estimado'] = sum(
    df_actual[c] * margenes.get(c, 0) for c in existing_prods
)

# KPI 1: Productos por cliente (PPP)
ppp_mean   = df_actual['n_productos'].mean()
ppp_median = df_actual['n_productos'].median()

# KPI 2: Cross-sell rate (clientes con 2 o más productos)
cross_sell_rate = (df_actual['n_productos'] >= 2).mean()

# KPI 3: Tasa de activación
tasa_activacion = df_actual['active_customer'].mean() if 'active_customer' in df_actual.columns else None

# KPI 4: Revenue estimado por cliente
rev_per_customer = df_actual['revenue_estimado'].mean()

# KPI 5: Penetración por producto
penetracion = df_actual[existing_prods].mean().sort_values(ascending=False)

print("📊 KPIs Estratégicos — Estado Actual (mayo 2019)")
print(f"  Total clientes:                 {len(df_actual):,}")
print(f"  Productos por cliente (media):  {ppp_mean:.2f}")
print(f"  Productos por cliente (mediana):{ppp_median:.0f}")
print(f"  Cross-sell rate (≥2 productos): {cross_sell_rate:.1%}")
if tasa_activacion is not None:
    print(f"  Tasa de activación en app:      {tasa_activacion:.1%}")
print(f"  Revenue estimado por cliente:   {rev_per_customer:.1f} €")
print(f"\nPenetración por producto:")
for prod, val in penetracion.items():
    print(f"  {prod:<25} {val:.1%}")

📊 KPIs Estratégicos — Estado Actual (mayo 2019)
  Total clientes:                 441,752
  Productos por cliente (media):  0.99
  Productos por cliente (mediana):1
  Cross-sell rate (≥2 productos): 14.3%
  Tasa de activación en app:      38.7%
  Revenue estimado por cliente:   13.7 €

Penetración por producto:
  em_acount                 67.0%
  debit_card                9.8%
  payroll_account           6.0%
  emc_account               5.6%
  pension_plan              3.9%
  payroll                   3.7%
  long_term_deposit         1.4%
  credit_card               1.1%
  securities                0.4%
  funds                     0.3%
  loans                     0.0%
  mortgage                  0.0%
  em_account_p              0.0%
  short_term_deposit        0.0%


In [33]:
# ============================================================
# VISUALIZACIÓN — PENETRACIÓN POR PRODUCTO Y KPIs CLAVE
# ============================================================

fig_kpi = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Penetración por Producto (%)',
        'KPIs Clave — Estado Actual'
    ],
    horizontal_spacing=0.15
)

# Gráfico 1: Penetración por producto
fig_kpi.add_trace(go.Bar(
    x=penetracion.values,
    y=penetracion.index,
    orientation='h',
    marker_color='#2196F3',
    text=[f"{v:.1%}" for v in penetracion.values],
    textposition='outside'
), row=1, col=1)

# Gráfico 2: KPIs clave como indicadores
kpi_nombres = ['PPP<br>(media)', 'Cross-sell<br>Rate', 'Activación<br>App', 'Revenue<br>/Cliente (€)']
kpi_valores = [ppp_mean, cross_sell_rate * 100, tasa_activacion * 100, rev_per_customer]
kpi_colores = ['#66BB6A', '#FFA726', '#2196F3', '#AB47BC']

fig_kpi.add_trace(go.Bar(
    x=kpi_nombres,
    y=kpi_valores,
    marker_color=kpi_colores,
    text=[f"{ppp_mean:.2f}", f"{cross_sell_rate:.1%}", f"{tasa_activacion:.1%}", f"{rev_per_customer:.1f} €"],
    textposition='outside'
), row=1, col=2)

fig_kpi.update_layout(
    title=dict(text='KPIs Estratégicos — easyMoney (mayo 2019)',
               x=0.5, font=dict(size=16)),
    template='plotly_white',
    showlegend=False,
    width=1100, height=500
)
fig_kpi.update_xaxes(range=[0, 0.80], row=1, col=1)
fig_kpi.show()

El gráfico de penetración revela el punto de partida de la estrategia: el **67% de los clientes solo tiene la cuenta básica** (`em_acount`) y apenas el **1.1% tiene `credit_card`** — lo que confirma el enorme recorrido de la campaña. Los cuatro KPIs clave (PPP 0.99, Cross-sell 14.3%, Activación 38.7%, Revenue 13.7 €/cliente) están todos por debajo de sus benchmarks objetivo, lo que define con precisión las palancas de mejora de la estrategia de penetración de cartera.

## 3.2 Evolución temporal de KPIs estratégicos

Analizamos la evolución mensual de los KPIs desde enero de 2018 hasta mayo de 2019 para identificar tendencias y el impacto de eventos clave en el negocio.

In [34]:
# ============================================================
# EVOLUCIÓN TEMPORAL DE KPIs ESTRATÉGICOS
# ============================================================

kpis_temporal = []

for partition, grp in df_master.groupby('pk_partition'):
    grp = grp.copy()
    existing_p = [c for c in existing_prods if c in grp.columns]
    grp['n_productos'] = grp[existing_p].sum(axis=1)
    grp['revenue_estimado'] = sum(
        grp[c] * margenes.get(c, 0) for c in existing_p
    )

    row = {
        'Particion':         partition,
        'N_Clientes':        len(grp),
        'PPP':               grp['n_productos'].mean(),
        'Cross_sell_Rate':   (grp['n_productos'] >= 2).mean(),
        'Revenue_Cliente':   grp['revenue_estimado'].mean(),
    }
    if 'active_customer' in grp.columns:
        row['Tasa_Activacion'] = grp['active_customer'].mean()

    kpis_temporal.append(row)

df_temp = pd.DataFrame(kpis_temporal).sort_values('Particion')
df_temp['Mes'] = df_temp['Particion'].dt.strftime('%Y-%m')

print("Evolución mensual de KPIs estratégicos:")
print(df_temp[['Mes', 'N_Clientes', 'PPP', 'Cross_sell_Rate', 'Tasa_Activacion', 'Revenue_Cliente']].to_string(index=False))

Evolución mensual de KPIs estratégicos:
    Mes  N_Clientes      PPP  Cross_sell_Rate  Tasa_Activacion  Revenue_Cliente
2018-01      238991 1.239172         0.156491         0.451293        16.530899
2018-02      242045 1.247867         0.160916         0.457535        16.779566
2018-03      244720 1.260293         0.167224         0.463395        16.998946
2018-04      246914 1.267753         0.171266         0.469410        17.134581
2018-05      249404 1.267033         0.172150         0.475826        17.125387
2018-06      251533 1.279427         0.176784         0.481531        17.335618
2018-07      335669 1.002747         0.137099         0.383649        13.511763
2018-08      351727 0.990970         0.130482         0.386428        13.244755
2018-09      372285 0.989675         0.129565         0.387488        13.319156
2018-10      399645 0.977510         0.128396         0.378891        13.263246
2018-11      414680 0.970150         0.127431         0.375878        13.163543


In [ ]:
# ============================================================
# VISUALIZACIÓN — EVOLUCIÓN TEMPORAL
# ============================================================

fig_temp = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Clientes Totales',
        'Productos por Cliente (PPP)',
        'Cross-sell Rate',
        'Revenue Estimado / Cliente (€)'
    ],
    vertical_spacing=0.22,
    horizontal_spacing=0.12
)

line_color = '#2196F3'

# Clientes totales
fig_temp.add_trace(go.Scatter(
    x=df_temp['Mes'], y=df_temp['N_Clientes'],
    mode='lines+markers', line=dict(color='#AB47BC', width=2),
    marker=dict(size=6), name='Clientes'
), row=1, col=1)

# PPP
fig_temp.add_trace(go.Scatter(
    x=df_temp['Mes'], y=df_temp['PPP'],
    mode='lines+markers', line=dict(color=line_color, width=2),
    marker=dict(size=6), name='PPP'
), row=1, col=2)

# Cross-sell rate
fig_temp.add_trace(go.Scatter(
    x=df_temp['Mes'], y=df_temp['Cross_sell_Rate'],
    mode='lines+markers', line=dict(color='#FFA726', width=2),
    marker=dict(size=6), name='Cross-sell'
), row=2, col=1)

# Revenue / cliente
fig_temp.add_trace(go.Scatter(
    x=df_temp['Mes'], y=df_temp['Revenue_Cliente'],
    mode='lines+markers', line=dict(color='#66BB6A', width=2),
    marker=dict(size=6), name='Revenue'
), row=2, col=2)

# Línea vertical en julio 2018 (inflexión)
for r, c in [(1,1),(1,2),(2,1),(2,2)]:
    fig_temp.add_vline(
        x='2018-07', line_dash='dash',
        line_color='#EF5350', line_width=1.5,
        row=r, col=c
    )

fig_temp.update_layout(
    title=dict(text='Evolución Mensual de KPIs Estratégicos (ene 2018 — may 2019)',
               x=0.5, font=dict(size=15)),
    template='plotly_white',
    showlegend=False,
    width=1050, height=580
)
fig_temp.update_xaxes(tickangle=-45)
fig_temp.show()

> ⚠️ **Nota — Punto de inflexión julio 2018:** La línea roja vertical marca el momento en que la base de clientes crece de golpe de 251.533 a 335.669 (+84.000 clientes nuevos). Estos clientes recién captados tienen, de media, menos productos y menor actividad, lo que deprime temporalmente el PPP, el Cross-sell Rate y el Revenue por cliente. Desde entonces se observa una **recuperación gradual y sostenida** en todos los indicadores.

## 3.3 Tabla resumen de KPIs — Propuesta para el Plan Estratégico

In [ ]:
# ============================================================
# TABLA RESUMEN DE KPIs — PROPUESTA PARA EL PLAN ESTRATÉGICO
# ============================================================

resumen_kpis = pd.DataFrame([
    # KPIs DE CAMPAÑA
    {
        'Categoría':    'Campaña',
        'KPI':          'Delivery Rate',
        'Definición':   'Emails entregados / Emails enviados',
        'Valor Actual': f"{total_row['Delivery Rate']:.1%}",
        'Benchmark':    f"{BENCHMARK_DELIVERY_RATE:.0%}",
        'Frecuencia':   'Por campaña',
        'Fuente':       'ESP'
    },
    {
        'Categoría':    'Campaña',
        'KPI':          'Open Rate',
        'Definición':   'Emails abiertos / Emails entregados',
        'Valor Actual': f"{total_row['Open Rate']:.1%}",
        'Benchmark':    f"{BENCHMARK_OPEN_RATE:.0%}",
        'Frecuencia':   'Por campaña',
        'Fuente':       'ESP'
    },
    {
        'Categoría':    'Campaña',
        'KPI':          'CTR',
        'Definición':   'Clicks / Emails entregados',
        'Valor Actual': f"{total_row['CTR']:.1%}",
        'Benchmark':    f"{BENCHMARK_CTR:.1%}",
        'Frecuencia':   'Por campaña',
        'Fuente':       'ESP'
    },
    {
        'Categoría':    'Campaña',
        'KPI':          'Tasa de Conversión',
        'Definición':   'Contratos firmados / Emails enviados',
        'Valor Actual': f"{total_row['Conv Rate']:.1%}",
        'Benchmark':    f"{BENCHMARK_CONV_RATE:.1%}",
        'Frecuencia':   'Por campaña',
        'Fuente':       'ESP + CRM'
    },
    {
        'Categoría':    'Campaña',
        'KPI':          'ROI de Campaña',
        'Definición':   '(Revenue - Costo) / Costo',
        'Valor Actual': f"{total_row['ROI (%)']:.0f}%",
        'Benchmark':    '>500%',
        'Frecuencia':   'Por campaña',
        'Fuente':       'ESP + Finanzas'
    },
    # KPIs ESTRATÉGICOS
    {
        'Categoría':    'Estratégico',
        'KPI':          'PPP (Productos por Cliente)',
        'Definición':   'Nº medio de productos por cliente activo',
        'Valor Actual': f"{ppp_mean:.2f}",
        'Benchmark':    '> 1.5',
        'Frecuencia':   'Mensual',
        'Fuente':       'CRM / BBDD'
    },
    {
        'Categoría':    'Estratégico',
        'KPI':          'Cross-sell Rate',
        'Definición':   'Clientes con ≥2 productos / Total clientes',
        'Valor Actual': f"{cross_sell_rate:.1%}",
        'Benchmark':    '> 25%',
        'Frecuencia':   'Mensual',
        'Fuente':       'CRM / BBDD'
    },
    {
        'Categoría':    'Estratégico',
        'KPI':          'Tasa de Activación',
        'Definición':   'Clientes activos en app / Total clientes',
        'Valor Actual': f"{tasa_activacion:.1%}",
        'Benchmark':    '> 50%',
        'Frecuencia':   'Mensual',
        'Fuente':       'App / CRM'
    },
    {
        'Categoría':    'Estratégico',
        'KPI':          'Revenue por Cliente',
        'Definición':   'Revenue estimado medio por cliente',
        'Valor Actual': f"{rev_per_customer:.1f} €",
        'Benchmark':    '> 20 €',
        'Frecuencia':   'Mensual',
        'Fuente':       'Finanzas / BBDD'
    },
    {
        'Categoría':    'Estratégico',
        'KPI':          'Retención mensual',
        'Definición':   'Clientes mes N / Clientes mes N-1',
        'Valor Actual': 'N/D',
        'Benchmark':    '> 98%',
        'Frecuencia':   'Mensual',
        'Fuente':       'CRM / BBDD'
    },
])

print("📋 Tabla de KPIs — Propuesta para Plan Estratégico easyMoney")
print("="*90)
print(resumen_kpis.to_string(index=False))
print("\n✅ Framework de KPIs definido.")

---
# 4. Conclusiones

## 4.1 KPIs de Campaña

- Los **3 KPIs clave** (Open Rate, CTR y Conv Rate) superan los benchmarks del sector financiero, lo que valida la estrategia de personalización por creatividad aplicada en la Tarea 4.
- La creatividad **"Premium y exclusivo"** obtiene los mejores resultados en todos los indicadores (Open Rate 26.8%, Conv Rate 2.2%).
- Con un **ROI estimado del 10.331%** (10.260 € de revenue frente a 98 € de coste), la campaña es altamente rentable incluso con tasas de conversión conservadoras.

## 4.2 KPIs Estratégicos

- El **PPP medio es de 0,99** (mediana: 1), lo que refleja que la mayoría de clientes tiene exactamente un producto activo (`em_acount`). El valor inferior a 1 indica que una fracción de los clientes registrados en la base de datos no tiene ningún producto activo en el periodo analizado — lo que representa el mayor potencial de cross-sell.
- El **Cross-sell Rate del 14,3%** está muy por debajo del benchmark objetivo (>25%). La campaña de `credit_card` es un primer paso concreto para mejorarlo.
- La **Tasa de Activación del 38,7%** indica que más de la mitad de los clientes no interactúa con la app, lo que representa otra palanca de mejora.
- El **punto de inflexión de julio 2018** (entrada masiva de ~84.000 nuevos clientes, de 251.533 a 335.669) explica la caída en todos los KPIs. Desde entonces se observa una recuperación gradual y sostenida.

## 4.3 Próximos pasos recomendados

| Acción | Impacto esperado |
|--------|-----------------|
| Ejecutar campaña credit_card (9.836 clientes) | +171 contratos, +10.260 € revenue |
| Campaña de reactivación (clientes inactivos) | ↑ Tasa de Activación |
| Segunda oleada cross-sell (fondos, pension_plan) | ↑ PPP, ↑ Revenue/cliente |
| Monitorizar KPIs mensualmente con este framework | Seguimiento estrategia Ansoff |